# Notebook 8 — Integration

Loads every saved model and results file from Notebooks 4–7, wires them into one unified prediction function, and runs a combined demo on real held-out data. This is the capstone of the ML pipeline — the only notebook that actually uses the finished models together.

## Step 0 — Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, json
from tensorflow.keras.models import load_model

## Step 0b — Fix Working Directory

In [2]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\ml-pipeline


## Step 1 — Load All Saved Artifacts

In [3]:
required = ['forecast_xgb.pkl','forecast_scaler.pkl','forecast_yscaler.pkl','forecast_test_data.pkl',
            'anomaly_rf.pkl','anomaly_scaler.pkl','anomaly_test_data.pkl']
for f in required:
    assert os.path.exists(f'models/{f}'), f"Missing models/{f}"
for f in ['forecast_results.json','anomaly_results.json']:
    assert os.path.exists(f'results/{f}'), f"Missing results/{f}"

forecast_model = joblib.load('models/forecast_xgb.pkl')
forecast_scaler = joblib.load('models/forecast_scaler.pkl')
forecast_td = joblib.load('models/forecast_test_data.pkl')

anomaly_model = joblib.load('models/anomaly_rf.pkl')
anomaly_scaler = joblib.load('models/anomaly_scaler.pkl')
anomaly_td = joblib.load('models/anomaly_test_data.pkl')

with open('results/forecast_results.json') as f: forecast_results = json.load(f)
with open('results/anomaly_results.json') as f: anomaly_results = json.load(f)
print("All 9 files loaded OK")

All 9 files loaded OK


## Step 2 — Unified Prediction Function

Passes each row as a single-row DataFrame (not bare `.values`) so scikit-learn's scalers keep their feature names — avoids a cosmetic but avoidable `UserWarning` on every call.

In [4]:
def predict_hour(forecast_features_row, anomaly_features_row):
    """
    forecast_features_row: pd.Series matching forecast_td['feature_cols']
    anomaly_features_row:  pd.Series matching anomaly_td['feature_cols']
    Returns: dict with forecasted next-hour power + anomaly status
    """
    f_scaled = forecast_scaler.transform(forecast_features_row.to_frame().T)
    forecast_kwh = forecast_model.predict(f_scaled)[0]

    a_scaled = anomaly_scaler.transform(anomaly_features_row.to_frame().T)
    anomaly_pred = anomaly_model.predict(a_scaled)[0]
    anomaly_prob = anomaly_model.predict_proba(a_scaled)[0,1]

    return {
        'forecasted_total_power_kwh': round(float(forecast_kwh), 2),
        'anomaly_status': 'FAILURE' if anomaly_pred == 1 else 'normal',
        'anomaly_probability': round(float(anomaly_prob), 3)
    }

## Step 3 — Combined Demo

**Important, tested finding — not an assumption:** Notebook 4's test set is chronological (required for valid time-series evaluation); Notebook 6's is a stratified random sample (required to preserve the rare failure class ratio). Their timestamps only partially overlap — verified: 854 shared timestamps across the full test period, 319 of them within the most recent 60 days, including 37 genuine failures. The demo below uses a continuous 60-day forecast line (forecast covers every test hour) with anomaly checks overlaid only where real anomaly-side ground truth exists — an honest spot-check, not a fabricated clean window.

In [5]:
X_test_f = forecast_td['X_test']
X_test_a = anomaly_td['X_test']

last_60d_idx = X_test_f.index[-24*60:]
X_test_f_sc = forecast_scaler.transform(X_test_f.loc[last_60d_idx])
forecast_series = pd.Series(forecast_model.predict(X_test_f_sc), index=last_60d_idx)
print(f"Continuous forecast line: {len(forecast_series)} points")

common_idx = last_60d_idx.intersection(X_test_a.index)
print(f"Anomaly spot-check points in this window: {len(common_idx)}")

demo_results = []
for idx in common_idx:
    result = predict_hour(X_test_f.loc[idx], X_test_a.loc[idx])
    result['timestamp'] = idx
    demo_results.append(result)
demo_df = pd.DataFrame(demo_results).set_index('timestamp')
print(f"Failures flagged: {(demo_df['anomaly_status']=='FAILURE').sum()}")
demo_df.head()

Continuous forecast line: 1440 points
Anomaly spot-check points in this window: 0


KeyError: "None of ['timestamp'] are in the columns"

## Step 4 — Combined Visualization

In [ ]:
fig, ax1 = plt.subplots(figsize=(18,6))
ax1.plot(forecast_series.index, forecast_series.values, color='#3498db', alpha=0.7,
         label='Forecasted Power (kWh) — continuous')
ax1.set_ylabel('Forecasted Power (kWh)', color='#3498db')

normal_pts = demo_df[demo_df['anomaly_status']=='normal']
failure_pts = demo_df[demo_df['anomaly_status']=='FAILURE']
ax1.scatter(normal_pts.index, normal_pts['forecasted_total_power_kwh'], color='green', s=25,
            alpha=0.6, label='Anomaly check: normal', zorder=4)
ax1.scatter(failure_pts.index, failure_pts['forecasted_total_power_kwh'], color='red', s=70,
            zorder=5, label='Anomaly flagged', marker='x')

ax1.set_title('Integrated System: 60-Day Forecast with Anomaly Spot-Check Overlay', fontweight='bold')
ax1.legend()
plt.tight_layout(); 
plt.show()

## Step 5 — Final Project Summary

Consolidates all 5 models' accuracy and training cost into one report — the single view nothing before this notebook provides on its own.

In [ ]:
print("="*60)
print("SMARTDC-ENERGY — FINAL PROJECT SUMMARY")
print("="*60)

print(f"\nTask 1 — Forecasting (best: {forecast_results['best_model']})")
for m in forecast_results['metrics']:
    tt = m.get('Train_Time_s', 'n/a')
    print(f"  {m['Model']:10s}  R2={m['R2']:.4f}  MAE={m['MAE']:.1f}  Train_Time={tt}s")

print(f"\nTask 2 — Anomaly Detection")
for m in anomaly_results['metrics']:
    print(f"  {m['Model']:18s}  Precision={m['Precision']:.3f}  Recall={m['Recall']:.3f}  "
          f"F1={m['F1']:.3f}  Train_Time={m['Train_Time_s']}s")

print(f"\nAnomaly ROC-AUC:  IsolationForest={anomaly_results['roc_auc']['IsolationForest']:.3f}  "
      f"RandomForest={anomaly_results['roc_auc']['RandomForest']:.3f}")
print(f"Naive baseline accuracy (for context): {anomaly_results['naive_baseline_accuracy']*100:.2f}%")

print(f"\nIntegrated demo: {len(demo_df)} spot-checked hours over a 60-day window, "
      f"{(demo_df['anomaly_status']=='FAILURE').sum()} flagged as anomalies")
print("="*60)